In [1]:
import json
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import ToolParam

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time as a formatted string. "
        "Use it whenever the user asks about today's date, the current time, "
        "or anything that depends on knowing 'now'. "
        "The format is controlled by the date_format argument and defaults to "
        "year-month-day hours:minutes:seconds."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "Python strftime format codes that set the output format, "
                    "for example '%Y-%m-%d' for a date only or '%H:%M' for the time only. Optional."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

In [3]:
def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if hasattr(message, "content") else message,
    })

def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if hasattr(message, "content") else message,
    })

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

In [4]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")

response = chat(messages, tools=[get_current_datetime_schema])
print(response.content)

[ToolUseBlock(id='toolu_01Lf6g7XCHWS6zoDiki3iqey', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]


In [5]:
for block in response.content:
    print(block.type)
    if block.type == "text":
        print("  text:", block.text)
    elif block.type == "tool_use":
        print("  id:", block.id)
        print("  name:", block.name)
        print("  input:", block.input)

print("stop_reason:", response.stop_reason)

tool_use
  id: toolu_01Lf6g7XCHWS6zoDiki3iqey
  name: get_current_datetime
  input: {'date_format': '%H:%M:%S'}
stop_reason: tool_use


In [6]:
add_assistant_message(messages, response)

## Find the `tool_use` block and run the function

In [8]:
tool_block = next(b for b in response.content if b.type == "tool_use")

result = get_current_datetime(**tool_block.input)
print(result)

02:11:46


## Send the result back

In [9]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": result,
        "is_error": False,
    }]
})

## The final request

In [10]:
final = chat(messages, tools=[get_current_datetime_schema])
print(text_from_message(final))

The exact time is **02:11:46** (HH:MM:SS format).
